<a href="https://colab.research.google.com/github/jiho1332/daython-energy-analysis/blob/main/IDC_peakW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

In [ ]:
df_columns = pd.read_csv('/content/drive/MyDrive/ZeroWatt/rtu_data_full.csv', nrows=0
)

df_columns.columns.tolist()

['module(equipment)', 'timestamp', 'localtime', 'operation', 'voltageR', 'voltageS', 'voltageT', 'voltageRS', 'voltageST', 'voltageTR', 'currentR', 'currentS', 'currentT', 'activePower', 'powerFactorR', 'powerFactorS', 'powerFactorT', 'reactivePowerLagging', 'accumActiveEnergy']


In [ ]:
# 1) 일단 헤더 + 상위 1000행만 빠르게 확인
df_sample = pd.read_csv('/content/drive/MyDrive/ZeroWatt/rtu_data_full.csv', nrows=1000)
print(df_sample.dtypes)
print(df_sample["module(equipment)"].unique())   # 설비 종류 몇 개인지
print(df_sample["operation"].unique())            # 운전상태 코드값 확인

# 2) chunksize로 전체 통계만 훑기 (메모리 절약)
chunks = pd.read_csv('/content/drive/MyDrive/ZeroWatt/rtu_data_full.csv', chunksize=500_000)
for chunk in chunks:
    print(chunk["module(equipment)"].value_counts())
    break  # 일단 1청크만 테스트

module(equipment)        object
timestamp                 int64
localtime                 int64
operation                 int64
voltageR                float64
voltageS                float64
voltageT                float64
voltageRS               float64
voltageST               float64
voltageTR               float64
currentR                float64
currentS                float64
currentT                float64
activePower             float64
powerFactorR            float64
powerFactorS            float64
powerFactorT            float64
reactivePowerLagging    float64
accumActiveEnergy         int64
dtype: object
['1(PM-3)']
[1]
module(equipment)
1(PM-3)    500000
Name: count, dtype: int64


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/ZeroWatt/rtu_data_full.csv'  # ← 여기만 본인 파일 경로로 수정!

# 1) timestamp 간격 확인 (앞 20행)
df = pd.read_csv(path, nrows=20)
print("=== timestamp / localtime 샘플 ===")
print(df[["timestamp", "localtime", "operation", "activePower", "accumActiveEnergy"]])

print("\n=== timestamp 간격(diff) ===")
print(df["timestamp"].diff())

# 2) 전체 파일에서 unique module / operation 값 (chunk로 누적 확인)
modules = set()
operations = set()
for chunk in pd.read_csv(path, chunksize=500_000, usecols=["module(equipment)", "operation"]):
    modules.update(chunk["module(equipment)"].unique())
    operations.update(chunk["operation"].unique())

print("\n=== 전체 module 종류 ===")
print(modules)
print("\n=== 전체 operation 코드 종류 ===")
print(operations)

=== timestamp / localtime 샘플 ===
        timestamp       localtime  operation  activePower  accumActiveEnergy
0   1733040000000  20241201000000          1      2961.61            1955004
1   1733040005000  20241201000005          1      3017.48            1955008
2   1733040010000  20241201000010          1      2408.01            1955011
3   1733040015000  20241201000015          1      3289.33            1955016
4   1733040020000  20241201000020          1      3069.31            1955020
5   1733040025000  20241201000025          1      1581.48            1955022
6   1733040030000  20241201000030          1      2452.03            1955026
7   1733040035000  20241201000035          1      3285.18            1955030
8   1733040040000  20241201000040          1      3603.09            1955035
9   1733040045000  20241201000045          1      3339.72            1955040
10  1733040050000  20241201000050          1      2754.95            1955044
11  1733040055000  20241201000055          

In [ ]:
print(df.head(0))
print(df.value_counts())
print(df.isnull().sum())

   module(equipment)      timestamp       localtime  operation  voltageR  \
0            1(PM-3)  1733040000000  20241201000000          1    214.38   
1            1(PM-3)  1733040005000  20241201000005          1    214.05   
2            1(PM-3)  1733040010000  20241201000010          1    215.79   
3            1(PM-3)  1733040015000  20241201000015          1    210.39   
4            1(PM-3)  1733040020000  20241201000020          1    216.71   
5            1(PM-3)  1733040025000  20241201000025          1    214.18   
6            1(PM-3)  1733040030000  20241201000030          1    210.19   
7            1(PM-3)  1733040035000  20241201000035          1    216.97   
8            1(PM-3)  1733040040000  20241201000040          1    215.83   
9            1(PM-3)  1733040045000  20241201000045          1    213.49   
10           1(PM-3)  1733040050000  20241201000050          1    211.01   
11           1(PM-3)  1733040055000  20241201000055          1    218.20   
12          

In [ ]:
import pandas as pd

stats = {}
for chunk in pd.read_csv(path, chunksize=500_000,
                          usecols=["module(equipment)", "activePower"]):
    for mod, group in chunk.groupby("module(equipment)"):
        if mod not in stats:
            stats[mod] = {"sum": 0, "count": 0, "max": 0}
        stats[mod]["sum"] += group["activePower"].sum()
        stats[mod]["count"] += len(group)
        stats[mod]["max"] = max(stats[mod]["max"], group["activePower"].max())

for mod, s in stats.items():
    print(f"{mod}: 평균 {s['sum']/s['count']:.1f} kW, 최대 {s['max']:.1f} kW, 행수 {s['count']}")

1(PM-3): 평균 3010.4 kW, 최대 5184.1 kW, 행수 2592001
2(L-1전등): 평균 3009.9 kW, 최대 5188.9 kW, 행수 2592001
3(분쇄기(2)): 평균 3009.6 kW, 최대 5171.2 kW, 행수 2592001
4(분쇄기(1)): 평균 3010.7 kW, 최대 5207.2 kW, 행수 2592001
5(좌측분전반): 평균 3010.3 kW, 최대 5175.7 kW, 행수 2592001
11(우측분전반1): 평균 3009.9 kW, 최대 5220.9 kW, 행수 2592001
12(4호기): 평균 3010.0 kW, 최대 5209.8 kW, 행수 2592001
13(3호기): 평균 3008.7 kW, 최대 5219.7 kW, 행수 2592001
14(2호기): 평균 3009.7 kW, 최대 5194.0 kW, 행수 2592001
15(예비건조기): 평균 3010.4 kW, 최대 5213.1 kW, 행수 2592001
16(호이스트): 평균 3009.8 kW, 최대 5193.0 kW, 행수 2592001
17(6호기): 평균 3009.9 kW, 최대 5184.3 kW, 행수 2592001
18(우측분전반2): 평균 3010.0 kW, 최대 5195.1 kW, 행수 2592001


In [ ]:
# 안전하게: 처음 10000개 timestamp만 비교
df = pd.read_csv(path, usecols=["module(equipment)", "timestamp", "activePower"], nrows=130000)
# (130000 ≈ 13개 설비 × 10000 timestamp, 단 파일이 설비별로 정렬되어 있다면 nrows 조절 필요)

In [ ]:
print(path)

/content/drive/MyDrive/ZeroWatt/rtu_data_full.csv


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/ZeroWatt/rtu_data_full.csv'  # 본인 경로로 수정

# 1) 안전하게 일부만 읽기 (처음 10000개 timestamp만 비교)
df = pd.read_csv(path, usecols=["module(equipment)", "timestamp", "activePower"], nrows=130000)

# 2) 혹시 모를 timestamp 중복/순서 문제 방지용으로 모듈별 개수 확인
print("=== 샘플 내 module별 행수 ===")
print(df["module(equipment)"].value_counts())

# 3) wide format으로 피벗 (행=timestamp, 열=설비, 값=activePower)
pivot = df.pivot_table(index="timestamp", columns="module(equipment)", values="activePower")

# 4) 설비 간 상관계수 행렬
print("\n=== 설비 간 activePower 상관계수 행렬 ===")
print(pivot.corr().round(3))

# 5) 첫 번째 설비를 기준으로 다른 설비들과 '완전히 같은 값'인 비율 확인
print("\n=== 동일 timestamp에서 값이 완전히 똑같은 비율 ===")
cols = pivot.columns.tolist()
ref = cols[0]
for c in cols[1:]:
    same_ratio = (pivot[ref] == pivot[c]).mean()
    print(f"{ref} vs {c}: 완전 동일 비율 {same_ratio:.2%}")

=== 샘플 내 module별 행수 ===
module(equipment)
1(PM-3)    130000
Name: count, dtype: int64

=== 설비 간 activePower 상관계수 행렬 ===
module(equipment)  1(PM-3)
module(equipment)         
1(PM-3)                1.0

=== 동일 timestamp에서 값이 완전히 똑같은 비율 ===


In [ ]:
import pandas as pd

# path = "실제파일경로.csv"
rows_per_module = 2_592_001  # 이전에 확인된 행수
n_sample = 10000  # 각 설비에서 가져올 샘플 행 수

# 13개 설비 목록 (순서는 파일에 등장한 순서와 같다고 가정)
modules_order = [
    "1(PM-3)", "2(L-1전등)", "3(분쇄기(2))", "4(분쇄기(1))", "5(좌측분전반)",
    "11(우측분전반1)", "12(4호기)", "13(3호기)", "14(2호기)",
    "15(예비건조기)", "16(호이스트)", "17(6호기)", "18(우측분전반2)"
]

samples = []
for i, mod in enumerate(modules_order):
    start_row = i * rows_per_module
    # 헤더(1줄) + 이전 설비들 건너뛰기
    chunk = pd.read_csv(
        path,
        skiprows=range(1, start_row + 1),  # 헤더 제외하고 start_row까지 건너뛰기
        nrows=n_sample,
        usecols=["module(equipment)", "timestamp", "activePower"]
    )
    samples.append(chunk)
    print(f"{mod}: 실제로 읽힌 module 값 = {chunk['module(equipment)'].unique()}")

df = pd.concat(samples, ignore_index=True)

# 피벗 + 상관계수
pivot = df.pivot_table(index="timestamp", columns="module(equipment)", values="activePower")

print("\n=== 설비 간 activePower 상관계수 행렬 ===")
print(pivot.corr().round(3))

print("\n=== 동일 timestamp에서 값이 완전히 똑같은 비율 (기준: PM-3) ===")
cols = pivot.columns.tolist()
ref = cols[0]
for c in cols[1:]:
    same_ratio = (pivot[ref] == pivot[c]).mean()
    print(f"{ref} vs {c}: 완전 동일 비율 {same_ratio:.2%}")

1(PM-3): 실제로 읽힌 module 값 = ['1(PM-3)']
2(L-1전등): 실제로 읽힌 module 값 = ['2(L-1전등)']
3(분쇄기(2)): 실제로 읽힌 module 값 = ['3(분쇄기(2))']
4(분쇄기(1)): 실제로 읽힌 module 값 = ['4(분쇄기(1))']
5(좌측분전반): 실제로 읽힌 module 값 = ['5(좌측분전반)']
11(우측분전반1): 실제로 읽힌 module 값 = ['11(우측분전반1)']
12(4호기): 실제로 읽힌 module 값 = ['12(4호기)']
13(3호기): 실제로 읽힌 module 값 = ['13(3호기)']
14(2호기): 실제로 읽힌 module 값 = ['14(2호기)']
15(예비건조기): 실제로 읽힌 module 값 = ['15(예비건조기)']
16(호이스트): 실제로 읽힌 module 값 = ['16(호이스트)']
17(6호기): 실제로 읽힌 module 값 = ['17(6호기)']
18(우측분전반2): 실제로 읽힌 module 값 = ['18(우측분전반2)']

=== 설비 간 activePower 상관계수 행렬 ===
module(equipment)  1(PM-3)  11(우측분전반1)  12(4호기)  13(3호기)  14(2호기)  15(예비건조기)  \
module(equipment)                                                              
1(PM-3)              1.000       0.002    0.025   -0.001    0.019     -0.009   
11(우측분전반1)           0.002       1.000   -0.010   -0.015    0.009      0.002   
12(4호기)              0.025      -0.010    1.000    0.005   -0.008     -0.006   
13(3호기)             -0.001   

In [ ]:
import pandas as pd

# 드라이브 경로가 아닌 로컬 경로에서 복사된 csv 파일을 읽어옵니다.
df = pd.read_csv('/content/rtu_data_full.csv')

# 데이터가 잘 불러와졌는지 상위 5개 행 확인
df.head()